# PEQ-Projektauswertung

Kopiere dieses Notebook in dein Projektverzeichnis, setze `PROJECT_FILE` und führe alle Zellen aus. Verwende einen `.xlsx`-Projektexport aus dem PEQ-Nachweistool mit gespeicherten Ergebnissen im Blatt `OUT`. In der Vorlage ist ein Beispielexport aus dem Repository vorbelegt. Relative Pfade beziehen sich auf das Arbeitsverzeichnis des Notebook-Kernels.

Für die Auswertung braucht der Kernel `peexcel[analysis]`. Quarto muss für den optionalen HTML-Export als `quarto` erreichbar sein.

In [ ]:
from pathlib import Path

PROJECT_FILE = Path("../data/exports/examples/Bsp_San_6G.xlsx")
EXPORT_HTML = None  # None fragt nach, False überspringt, True fragt auch bei Überschreiben


## Projekt und Varianten

PEXL liest die gespeicherten Jahresergebnisse. Die Varianten werden gemeinsam und in der Reihenfolge des Projektexports dargestellt.

In [ ]:
from html import escape

import pandas as pd
from IPython.display import HTML, Markdown, display
from pexl import Project, reporting
from pexl.plot import reports

project_file = PROJECT_FILE.expanduser().resolve()
project = Project.from_excel(project_file)
if not len(project):
    raise ValueError(f"Keine Varianten im Projektexport {project_file.name} gefunden.")

variants = pd.DataFrame({"Variante": [scenario.column_name for scenario in project]})
project_summary = (
    f"<p>Projektexport <strong>{escape(project_file.name)}</strong> · "
    f"{len(project)} Variante(n)</p>"
)
variant_table = variants.to_html(index=False, escape=True, border=0, classes="table")
display(HTML(project_summary + variant_table))


## Jahresbilanzen

Die interaktiven Plotly-Diagramme enthalten alle Varianten. Über die Legende kannst du Beiträge ausblenden, per Maus Werte ablesen und den Ausschnitt vergrößern. `figures` enthält die fertigen Plotly-Figuren und `report_data` die zugehörigen Berichtsdaten. Damit kannst du einzelne Diagramme später für Präsentationen weiterverwenden.

Ist ein Diagramm wegen fehlender Ergebnisse oder eines Platzhalters im Berichtskatalog nicht darstellbar, erscheint an seiner Stelle ein Hinweis. Die übrigen Diagramme laufen weiter.

Die Primärenergiebilanz beschriftet standardmäßig die Differenz zwischen Deckung und Bedarf. Die THG-Übersicht zeigt eine gestapelte Säule je Variante, eine Raute für die gespeicherte Lebenszyklus-Gesamtbilanz und den Abstand zum Referenzbudget. `reports.render(data, annotate=False)` blendet die Differenzbeschriftungen aus. Für die THG-Übersicht setzt `reports.render(data, budget=250)` bei Bedarf ein eigenes Budget in der Einheit des Berichts.

In [ ]:
titles = {
    "heat_balance": "Heizwärmebilanz",
    "final_energy_balance": "Endenergiebilanz",
    "primary_energy_balance": "Primärenergiebilanz",
    "gwp_ee": "Treibhausgasbilanz",
    "gwp_ee_detail": "Treibhausgasbilanz im Detail",
    "gwp_ee_fossil_biogenic": "Graue Emissionen nach fossilen und biogenen Anteilen"
}
chart_names = reporting.chart_names()
ordered_charts = [name for name in titles if name in chart_names]
ordered_charts.extend(name for name in chart_names if name not in titles)
report_data = {}
figures = {}
notes = {}
section_titles = {}

for name in ordered_charts:
    title = titles.get(name, name)
    try:
        data = reporting.materialize(project, name, language="de")
        report_data[name] = data
        title = titles.get(name, data["title"])
        figure = reports.render(data, legend_position="right")
        if figure.data:
            figures[name] = figure
        else:
            notes[name] = "Für dieses Diagramm liefert PEXL derzeit keine darstellbaren Datenserien."
    except (ValueError, KeyError) as error:
        notes[name] = f"PEXL konnte dieses Diagramm nicht erstellen. {error}"

    section_titles[name] = title
    display(Markdown(f"### {title}"))
    if name in figures:
        figures[name].show(config={"responsive": True})
    else:
        display(HTML(f"<p>{escape(notes[name])}</p>"))


## HTML-Bericht speichern

Die letzte Zelle fragt standardmäßig nach. Mit `j` schreibt Quarto `<Projektdateiname>_Ergebnisse.html` ins Arbeitsverzeichnis des Notebook-Kernels, üblicherweise neben das Notebook. Bei einer vorhandenen Datei nennt die Abfrage ausdrücklich das Überschreiben. Mit `EXPORT_HTML = False` kannst du den Export für automatische Notebook-Läufe überspringen. `EXPORT_HTML = True` exportiert automatisch, sofern die Zieldatei noch frei ist.

Der Bericht übernimmt die bereits erzeugten Diagramme und Hinweise, mit Inhaltsverzeichnis und eingebetteten Plotly-Ressourcen für die Offline-Nutzung.

In [ ]:
import shutil
import subprocess
from tempfile import TemporaryDirectory

output_file = Path.cwd() / f"{project_file.stem}_Ergebnisse.html"
export = EXPORT_HTML
if export is None or (export and output_file.exists()):
    action = "überschreiben" if output_file.exists() else "speichern"
    try:
        export = input(f"HTML-Bericht {output_file.name} {action}? [j/N] ").strip().lower() in {"j", "ja"}
    except (EOFError, NotImplementedError):
        export = False

if export:
    quarto = shutil.which("quarto")
    if quarto is None:
        raise RuntimeError("Quarto ist im PATH des Notebook-Kernels nicht verfügbar.")

    sections = ["""---
title: PEQ-Projektauswertung
lang: de
format:
  html:
    toc: true
    toc-title: Inhalt
    embed-resources: true
---

## Projekt und Varianten
""", "```{=html}\n" + project_summary + variant_table + "\n```", "## Jahresbilanzen"]
    include_plotly = True
    for name, title in section_titles.items():
        sections.append(f"### {title}")
        if name in figures:
            content = figures[name].to_html(
                full_html=False, include_plotlyjs=include_plotly, config={"responsive": True}
            )
            include_plotly = False
        else:
            content = f"<p>{escape(notes[name])}</p>"
        sections.append(f"```{{=html}}\n{content}\n```")

    with TemporaryDirectory(prefix="pexl-ergebnisse-") as directory:
        source = Path(directory) / "ergebnisse.qmd"
        source.write_text("\n\n".join(sections), encoding="utf-8")
        subprocess.run([quarto, "render", source.name, "--to", "html"], cwd=directory, check=True)
        shutil.copyfile(source.with_suffix(".html"), output_file)
    print(f"Gespeichert unter {output_file}")
else:
    print("HTML-Export übersprungen.")
